# Module 4: Train MobileNetV2 Surrogate

This notebook is the second split-out stage of Module 4. It trains the MobileNetV2 surrogate with centralized supervised learning on the attacker data view used for input-level adversarial examples and black-box transfer evaluation.

Run this notebook from the `4_Adversarial_FL/` directory.


## Stage Goal

Train a differentiable attacker model without using MobileNetV3 target gradients. The handoff artifact is `artifacts/module4_surrogate.pt`, which the focused attack notebooks load to craft FGSM and PGD examples.


## 1. Notebook Setup

Load the surrogate-training utilities. This notebook uses ordinary supervised optimization and does not construct a federated server.


In [1]:
from contextlib import nullcontext
from copy import deepcopy
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "4_Adversarial_FL").exists():
    MODULE_DIR = MODULE_DIR / "4_Adversarial_FL"
SRC_DIR = MODULE_DIR / "src"
for path in (MODULE_DIR.parent, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

import json
import math
import yaml
import numpy as np
import torch
import torchvision.transforms as tv_transforms
import matplotlib.pyplot as plt
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from model import MobileNetV2Transfer
from util_functions import create_data, evaluate_fn, select_validation_subset, set_seed
from load_data_for_clients import dist_data_per_client


## 2. Configuration and Artifact Setup

Load `train_surrogate_config.yaml`, resolve the selected `surrogate_training.profile`, validate the centralized surrogate-training settings, and write a stage-specific config snapshot.

The surrogate now follows the same training-contract shape as the target notebook: a selected profile is merged with explicit per-run overrides, then one centralized MobileNetV2 checkpoint is saved for downstream attacks.


In [2]:
CONFIG_PATH = Path("train_surrogate_config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("Could not locate train_surrogate_config.yaml in the working directory")
with CONFIG_PATH.open() as f:
    CONFIG = yaml.safe_load(f)


def _merge_dicts(base: dict | None, override: dict | None) -> dict:
    merged = deepcopy(base or {})
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = deepcopy(value)
    return merged


def _resolve_surrogate_training_config(config: dict) -> tuple[str, dict]:
    selection = deepcopy(config.get("surrogate_training", {}))
    profiles = deepcopy(config.get("surrogate_training_profiles", {}))
    default_profile = "quick"
    profile = str(selection.get("profile", default_profile))
    if profile not in profiles:
        available = ", ".join(sorted(profiles)) or "none"
        raise ValueError(
            f"Unknown surrogate_training.profile={profile!r}. Available profiles: {available}."
        )

    cfg = _merge_dicts(profiles.get(profile, {}), selection)
    optimizer_cfg = cfg.get("optimizer", {}) if isinstance(cfg.get("optimizer", {}), dict) else {}
    criterion_cfg = cfg.get("criterion", {}) if isinstance(cfg.get("criterion", {}), dict) else {}

    if "lr" in optimizer_cfg:
        cfg["learning_rate"] = optimizer_cfg["lr"]
        cfg["lr"] = optimizer_cfg["lr"]
    if "weight_decay" in optimizer_cfg:
        cfg["weight_decay"] = optimizer_cfg["weight_decay"]
    if "label_smoothing" in criterion_cfg:
        cfg["label_smoothing"] = criterion_cfg["label_smoothing"]
    return profile, cfg


global_config = deepcopy(CONFIG.get("global_config", {}))
data_config = deepcopy(CONFIG.get("data_config", {}))
model_config = deepcopy(CONFIG.get("model_config", {}))
alg_configs = deepcopy(CONFIG.get("algorithms", {}))
artifact_config = deepcopy(CONFIG.get("artifacts", {}))
surrogate_training_profiles = deepcopy(CONFIG.get("surrogate_training_profiles", {}))
SURROGATE_PROFILE, surrogate_training_config = _resolve_surrogate_training_config(CONFIG)
DEFAULT_FED_CONFIG = deepcopy(alg_configs.get("FedAvg", {}).get("fed_config", {}))
NUM_CLASSES = int(model_config.get("kwargs", {}).get("num_classes", 10))
set_seed(global_config.get("seed", 42))


def get_device(preferred: str | None = None) -> torch.device:
    choice = preferred if preferred is not None else global_config.get("device")
    if isinstance(choice, str):
        if choice.startswith("cuda") and not torch.cuda.is_available():
            choice = "cpu"
        return torch.device(choice)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
global_config["device"] = str(DEVICE)

ARTIFACT_DIR = Path(artifact_config.get("dir", "artifacts"))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def artifact_path(key: str, default_filename: str) -> Path:
    return ARTIFACT_DIR / artifact_config.get(key, default_filename)


def validate_surrogate_config() -> None:
    issues = []
    optimizer_cfg = surrogate_training_config.get("optimizer", {}) if isinstance(surrogate_training_config.get("optimizer", {}), dict) else {}
    scheduler_cfg = surrogate_training_config.get("scheduler", {}) if isinstance(surrogate_training_config.get("scheduler", {}), dict) else {}
    criterion_cfg = surrogate_training_config.get("criterion", {}) if isinstance(surrogate_training_config.get("criterion", {}), dict) else {}
    optimizer_name = str(optimizer_cfg.get("name", "sgd")).lower()
    scheduler_name = str(scheduler_cfg.get("name", "none")).lower()
    mode = str(surrogate_training_config.get("mode", "centralized")).lower()
    data_view = str(surrogate_training_config.get("data_view", "client_pool")).lower()
    num_epochs = int(surrogate_training_config.get("num_epochs", 0))
    batch_size = int(surrogate_training_config.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
    pool_size = int(surrogate_training_config.get("pool_size", 1))
    label_smoothing = float(criterion_cfg.get("label_smoothing", surrogate_training_config.get("label_smoothing", 0.0)))

    if mode != "centralized":
        issues.append("surrogate_training.mode must be 'centralized'.")
    if data_view not in {"client_pool", "full_train"}:
        issues.append("surrogate_training.data_view must be one of: client_pool, full_train.")
    if not surrogate_training_config.get("dataset_path", data_config.get("dataset_path")):
        issues.append("surrogate_training.dataset_path or data_config.dataset_path is required.")
    if not surrogate_training_config.get("dataset_name", data_config.get("dataset_name")):
        issues.append("surrogate_training.dataset_name or data_config.dataset_name is required.")
    if num_epochs <= 0:
        issues.append("surrogate_training.num_epochs must be positive.")
    if batch_size <= 0:
        issues.append("surrogate_training.batch_size must be positive.")
    if data_view == "client_pool" and pool_size <= 0:
        issues.append("surrogate_training.pool_size must be positive when data_view=client_pool.")
    if optimizer_name not in {"sgd", "local_sgd", "adam", "adamw"}:
        issues.append("surrogate_training.optimizer.name must be one of: sgd, local_sgd, adam, adamw.")
    if scheduler_name not in {"none", "cosine", "warmup_cosine", "plateau", "reduce_on_plateau"}:
        issues.append("surrogate_training.scheduler.name must be one of: none, cosine, warmup_cosine, plateau, reduce_on_plateau.")
    if scheduler_name == "warmup_cosine" and int(scheduler_cfg.get("warmup_epochs", 0)) < 0:
        issues.append("surrogate_training.scheduler.warmup_epochs cannot be negative.")
    if not 0.0 <= label_smoothing < 1.0:
        issues.append("surrogate_training.criterion.label_smoothing must be in [0, 1).")
    if issues:
        raise ValueError("Surrogate-training config validation failed:\n" + "\n".join(issues))

    lr = float(optimizer_cfg.get("lr", optimizer_cfg.get("learning_rate", surrogate_training_config.get("learning_rate", 1e-3))))
    print(
        "Centralized surrogate config ready: "
        f"profile={SURROGATE_PROFILE}, device={DEVICE}, "
        f"dataset={surrogate_training_config.get('dataset_name', data_config.get('dataset_name'))}, "
        f"data_view={data_view}, epochs={num_epochs}, batch_size={batch_size}, "
        f"optimizer={optimizer_name}, scheduler={scheduler_name}, lr={lr:.3g}"
    )


validate_surrogate_config()

config_snapshot = deepcopy(CONFIG)
config_snapshot.setdefault("global_config", {})["resolved_device"] = str(DEVICE)
config_snapshot.setdefault("surrogate_training", {})["selected_profile"] = SURROGATE_PROFILE
config_snapshot.setdefault("surrogate_training", {})["resolved_mode"] = "centralized"
config_snapshot["surrogate_training_effective"] = deepcopy(surrogate_training_config)
config_path = artifact_path("config_snapshot", "module4_surrogate_config_used.json")
with config_path.open("w") as f:
    json.dump(config_snapshot, f, indent=2)

print("Loaded config from", CONFIG_PATH.resolve())
print(f"Saved surrogate config snapshot to {config_path.resolve()}")


Centralized surrogate config ready: profile=tuned_imagenette, device=cuda, dataset=Imagenette, data_view=client_pool, epochs=300, batch_size=64, optimizer=sgd, scheduler=warmup_cosine, lr=0.004
Loaded config from /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/train_surrogate_config.yaml
Saved surrogate config snapshot to /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_surrogate_config_used.json


## 3. Centralized Surrogate Data

Build the configured centralized attacker data view. The default `client_pool` view trains one MobileNetV2 surrogate over pooled client shards; `full_train` is available only when intentionally modeling an attacker with access to the full training set.


In [3]:
SURROGATE_CFG = surrogate_training_config
SURROGATE_DATA_VIEW = str(SURROGATE_CFG.get("data_view", "client_pool")).lower()
SURROGATE_CLIENT_ID = int(SURROGATE_CFG.get("client_id", 0))
SURROGATE_SEED = SURROGATE_CFG.get("seed", global_config.get("seed", 42))
SURROGATE_POOL_SIZE = max(1, int(SURROGATE_CFG.get("pool_size", 1)))
SURROGATE_NUM_CLIENTS = int(SURROGATE_CFG.get("num_clients", DEFAULT_FED_CONFIG.get("num_clients", 50)))
SURROGATE_BATCH_SIZE = int(SURROGATE_CFG.get("batch_size", DEFAULT_FED_CONFIG.get("batch_size", 64)))
SURROGATE_EVAL_BATCH_SIZE = int(SURROGATE_CFG.get("eval_batch_size", 512))
SURROGATE_EVAL_SUBSET = str(SURROGATE_CFG.get("eval_subset", data_config.get("eval_subset", "selection")))

_SURROGATE_CLIENT_LOADERS = None
_SURROGATE_TRAIN_LOADER = None
_SURROGATE_TEST_LOADER = None
SURROGATE_SELECTED_CLIENT_IDS: list[int] = []


class AugmentedDataset(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        image, label = self.base_dataset[index]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def _surrogate_transform():
    if not SURROGATE_CFG.get("augment", False):
        return None
    return tv_transforms.Compose([
        tv_transforms.RandomHorizontalFlip(),
        tv_transforms.RandomRotation(degrees=10),
    ])


def _maybe_augmented(dataset):
    transform = _surrogate_transform()
    if transform is None:
        return dataset
    return AugmentedDataset(dataset, transform=transform)


def _build_train_loader(dataset) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=SURROGATE_BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=int(SURROGATE_CFG.get("num_workers", 0)),
        pin_memory=bool(SURROGATE_CFG.get("pin_memory", DEVICE.type == "cuda")),
    )


def _build_eval_loader(dataset) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=SURROGATE_EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=int(SURROGATE_CFG.get("num_workers", 0)),
        pin_memory=bool(SURROGATE_CFG.get("pin_memory", DEVICE.type == "cuda")),
    )


def _ensure_surrogate_data():
    global _SURROGATE_CLIENT_LOADERS, _SURROGATE_TRAIN_LOADER, _SURROGATE_TEST_LOADER, SURROGATE_SELECTED_CLIENT_IDS
    if _SURROGATE_TRAIN_LOADER is not None and _SURROGATE_TEST_LOADER is not None:
        return _SURROGATE_TRAIN_LOADER, _SURROGATE_TEST_LOADER

    dataset_path = SURROGATE_CFG.get("dataset_path", data_config.get("dataset_path"))
    dataset_name = SURROGATE_CFG.get("dataset_name", data_config.get("dataset_name"))

    if SURROGATE_DATA_VIEW == "full_train":
        train_dataset, full_val_dataset = create_data(dataset_path, dataset_name)
        validation_dataset = select_validation_subset(
            full_val_dataset,
            data_config.get("validation_split"),
            SURROGATE_EVAL_SUBSET,
        )
        SURROGATE_SELECTED_CLIENT_IDS = []
        _SURROGATE_TRAIN_LOADER = _build_train_loader(_maybe_augmented(train_dataset))
        _SURROGATE_TEST_LOADER = _build_eval_loader(validation_dataset)
        return _SURROGATE_TRAIN_LOADER, _SURROGATE_TEST_LOADER

    client_loaders, test_loader = dist_data_per_client(
        data_path=dataset_path,
        dataset_name=dataset_name,
        num_clients=SURROGATE_NUM_CLIENTS,
        batch_size=SURROGATE_BATCH_SIZE,
        non_iid_per=SURROGATE_CFG.get("non_iid_per", data_config.get("non_iid_per", 0.0)),
        device=DEVICE,
        validation_split=data_config.get("validation_split"),
        eval_subset=SURROGATE_EVAL_SUBSET,
        seed=int(global_config.get("seed", 42)),
    )
    _SURROGATE_CLIENT_LOADERS = client_loaders
    pool_size = min(SURROGATE_POOL_SIZE, len(client_loaders))
    if SURROGATE_CLIENT_ID >= len(client_loaders):
        raise IndexError(
            f"Surrogate client id {SURROGATE_CLIENT_ID} out of range for {len(client_loaders)} clients"
        )
    SURROGATE_SELECTED_CLIENT_IDS = [
        int((SURROGATE_CLIENT_ID + offset) % len(client_loaders))
        for offset in range(pool_size)
    ]
    datasets = []
    for idx in SURROGATE_SELECTED_CLIENT_IDS:
        dataset = getattr(client_loaders[idx], "dataset", None)
        if dataset is None:
            raise ValueError("Client loader is missing a dataset attribute; cannot build surrogate pool")
        datasets.append(dataset)

    pooled_dataset = _maybe_augmented(ConcatDataset(datasets))
    _SURROGATE_TRAIN_LOADER = _build_train_loader(pooled_dataset)
    _SURROGATE_TEST_LOADER = _build_eval_loader(test_loader.dataset)
    return _SURROGATE_TRAIN_LOADER, _SURROGATE_TEST_LOADER


def get_surrogate_train_loader(pool: bool = True):
    train_loader, _ = _ensure_surrogate_data()
    if not pool and SURROGATE_DATA_VIEW == "client_pool":
        if _SURROGATE_CLIENT_LOADERS is None:
            raise RuntimeError("Client loaders have not been initialised.")
        return _SURROGATE_CLIENT_LOADERS[SURROGATE_CLIENT_ID]
    return train_loader


def get_surrogate_test_loader():
    _, test_loader = _ensure_surrogate_data()
    return test_loader


surrogate_train_loader = get_surrogate_train_loader(pool=True)
surrogate_test_loader = get_surrogate_test_loader()
print(
    f"Centralized surrogate data ready: data_view={SURROGATE_DATA_VIEW}, "
    f"train_examples={len(surrogate_train_loader.dataset)}, "
    f"selected_client_ids={SURROGATE_SELECTED_CLIENT_IDS or 'full_train'}, "
    f"validation_subset={SURROGATE_EVAL_SUBSET}, "
    f"validation_examples={len(surrogate_test_loader.dataset)}"
)



Preparing data with Dirichlet partitioner (aligned with Module 2)
Saved client data to cache: cache/client_data_570b222f889d10b8177996aa63e618ce.pkl
Centralized surrogate data ready: data_view=client_pool, train_examples=760, selected_client_ids=[0, 1, 2, 3], validation_subset=selection, validation_examples=1963


## 4. Surrogate Training Helpers

Define the MobileNetV2 model, loss, optimizer, scheduler, AMP helpers, and supervised training loop. This mirrors the target notebook's centralized training structure while keeping the surrogate architecture and attacker data view explicit.


In [4]:
def build_surrogate_model(num_classes: int = 10, pretrained: bool | None = None) -> torch.nn.Module:
    if pretrained is None:
        pretrained = SURROGATE_CFG.get("pretrained", True)
    return MobileNetV2Transfer(pretrained=pretrained, num_classes=num_classes)


def build_surrogate_criterion() -> torch.nn.Module:
    criterion_cfg = SURROGATE_CFG.get("criterion", {}) if isinstance(SURROGATE_CFG.get("criterion", {}), dict) else {}
    label_smoothing = float(criterion_cfg.get("label_smoothing", SURROGATE_CFG.get("label_smoothing", 0.0)))
    return torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing).to(DEVICE)


def build_surrogate_optimizer(model: torch.nn.Module) -> torch.optim.Optimizer:
    optimizer_cfg = SURROGATE_CFG.get("optimizer", {}) if isinstance(SURROGATE_CFG.get("optimizer", {}), dict) else {}
    name = str(optimizer_cfg.get("name", SURROGATE_CFG.get("optimizer_name", "sgd"))).lower()
    if name == "local_sgd":
        name = "sgd"
    lr = float(optimizer_cfg.get("lr", SURROGATE_CFG.get("learning_rate", SURROGATE_CFG.get("lr", 1e-3))))
    weight_decay = float(optimizer_cfg.get("weight_decay", SURROGATE_CFG.get("weight_decay", 0.0)))
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise RuntimeError("No trainable parameters available for surrogate optimisation.")

    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        momentum = float(optimizer_cfg.get("momentum", SURROGATE_CFG.get("momentum", 0.0)))
        return torch.optim.SGD(trainable_params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(f"Unsupported surrogate optimizer: {name}")


class WarmupCosineScheduler:
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        total_epochs: int,
        warmup_epochs: int,
        min_lr: float,
        warmup_start_factor: float = 0.1,
    ):
        self.optimizer = optimizer
        self.total_epochs = max(int(total_epochs), 1)
        self.warmup_epochs = max(int(warmup_epochs), 0)
        self.min_lr = float(min_lr)
        self.warmup_start_factor = float(warmup_start_factor)
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]
        self.epoch = 0
        if self.warmup_epochs > 0:
            self._set_lrs([lr * self.warmup_start_factor for lr in self.base_lrs])

    def _set_lrs(self, lrs):
        for group, lr in zip(self.optimizer.param_groups, lrs):
            group["lr"] = float(lr)

    def step(self):
        self.epoch += 1
        if self.warmup_epochs > 0 and self.epoch <= self.warmup_epochs:
            progress = self.epoch / max(self.warmup_epochs, 1)
            factor = self.warmup_start_factor + (1.0 - self.warmup_start_factor) * progress
            self._set_lrs([lr * factor for lr in self.base_lrs])
            return

        decay_epochs = max(self.total_epochs - self.warmup_epochs, 1)
        decay_step = min(max(self.epoch - self.warmup_epochs, 0), decay_epochs)
        cosine = 0.5 * (1.0 + math.cos(math.pi * decay_step / decay_epochs))
        self._set_lrs([
            self.min_lr + (base_lr - self.min_lr) * cosine
            for base_lr in self.base_lrs
        ])


def build_surrogate_scheduler(optimizer: torch.optim.Optimizer, epochs: int):
    scheduler_cfg = SURROGATE_CFG.get("scheduler", {}) if isinstance(SURROGATE_CFG.get("scheduler", {}), dict) else {}
    name = str(scheduler_cfg.get("name", "none")).lower()
    if name == "none":
        return None
    if name == "cosine":
        min_lr = float(scheduler_cfg.get("min_lr", scheduler_cfg.get("eta_min", 0.0)))
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(int(epochs), 1),
            eta_min=min_lr,
        )
    if name == "warmup_cosine":
        return WarmupCosineScheduler(
            optimizer,
            total_epochs=epochs,
            warmup_epochs=int(scheduler_cfg.get("warmup_epochs", 5)),
            min_lr=float(scheduler_cfg.get("min_lr", scheduler_cfg.get("eta_min", 0.0))),
            warmup_start_factor=float(scheduler_cfg.get("warmup_start_factor", 0.1)),
        )
    if name in {"plateau", "reduce_on_plateau"}:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=float(scheduler_cfg.get("factor", 0.5)),
            patience=int(scheduler_cfg.get("patience", 1)),
        )
    raise ValueError(f"Unsupported surrogate scheduler: {name}")


def _step_surrogate_scheduler(scheduler, val_loss: float) -> None:
    if scheduler is None:
        return
    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        scheduler.step(val_loss)
    else:
        scheduler.step()


def build_surrogate_amp(requested: bool):
    enabled = bool(requested) and DEVICE.type == "cuda"
    if requested and not enabled:
        print("AMP requested but disabled because the resolved device is not CUDA.")
    scaler = None
    if enabled:
        if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
            try:
                scaler = torch.amp.GradScaler("cuda", enabled=True)
            except TypeError:
                scaler = torch.amp.GradScaler(enabled=True)
        else:
            scaler = torch.cuda.amp.GradScaler(enabled=True)
    return enabled, scaler


def surrogate_amp_autocast(enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        try:
            return torch.amp.autocast(device_type="cuda", enabled=True)
        except TypeError:
            pass
    return torch.cuda.amp.autocast(enabled=True)


def train_surrogate_baseline(num_epochs: int | None = None):
    set_seed(global_config.get("seed", 42))
    train_loader = get_surrogate_train_loader(pool=True)
    test_loader = get_surrogate_test_loader()
    num_classes = SURROGATE_CFG.get("num_classes", NUM_CLASSES)
    model = build_surrogate_model(num_classes=num_classes).to(DEVICE)
    if SURROGATE_CFG.get("freeze_backbone", False) and hasattr(model, "v2model"):
        for param in model.v2model.features.parameters():
            param.requires_grad = False
    criterion = build_surrogate_criterion()
    optimizer = build_surrogate_optimizer(model)
    epochs = int(num_epochs or SURROGATE_CFG.get("num_epochs", 5))
    scheduler = build_surrogate_scheduler(optimizer, epochs)
    patience = int(SURROGATE_CFG.get("early_stop_patience", 0))
    use_amp, scaler = build_surrogate_amp(bool(SURROGATE_CFG.get("use_amp", False)))
    history = {"loss": [], "accuracy": [], "top1_accuracy": [], "val_loss": [], "val_accuracy": [], "val_top1_accuracy": [], "lr": []}
    best_state = None
    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    best_checkpoint_epoch = 0
    best_checkpoint_val_accuracy = 0.0
    epochs_since_improved = 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0
        epoch_lr = float(optimizer.param_groups[0]["lr"])
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE).float()
            labels = labels.to(DEVICE).long()
            optimizer.zero_grad(set_to_none=True)
            with surrogate_amp_autocast(use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()

        epoch_loss = running_loss / max(total, 1)
        epoch_acc = 100.0 * correct / max(total, 1)
        val_loss, val_acc = evaluate_fn(test_loader, model, criterion, DEVICE)
        _step_surrogate_scheduler(scheduler, val_loss)
        history["loss"].append(float(epoch_loss))
        history["accuracy"].append(float(epoch_acc))
        history["top1_accuracy"].append(float(epoch_acc))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_acc))
        history["val_top1_accuracy"].append(float(val_acc))
        history["lr"].append(epoch_lr)
        best_val_accuracy = max(best_val_accuracy, float(val_acc))

        improved = val_loss + 1e-5 < best_val_loss
        if improved:
            best_val_loss = float(val_loss)
            best_checkpoint_epoch = epoch + 1
            best_checkpoint_val_accuracy = float(val_acc)
            best_state = deepcopy(model.state_dict())
            epochs_since_improved = 0
            status = "saved best checkpoint"
        else:
            epochs_since_improved += 1
            status = (
                f"no val-loss improvement ({epochs_since_improved}/{patience})"
                if patience
                else "no val-loss improvement"
            )

        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"train_loss={epoch_loss:.4f}, train_acc={epoch_acc:.2f}%, "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.2f}%, "
            f"lr={epoch_lr:.2e} - {status}"
        )
        if not improved and patience and epochs_since_improved >= patience:
            print(f"Stopping early at epoch {epoch + 1} after {patience} epochs without validation-loss improvement.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_acc = evaluate_fn(test_loader, model, criterion, DEVICE)
    last_train_acc = float(history["accuracy"][-1]) if history["accuracy"] else 0.0
    final_val_acc = float(test_acc)
    train_val_gap = float(last_train_acc - final_val_acc)
    summary = {
        "training_mode": "centralized",
        "surrogate_training_profile": SURROGATE_PROFILE,
        "profile_description": SURROGATE_CFG.get("description", ""),
        "model": "MobileNetV2Transfer",
        "dataset": SURROGATE_CFG.get("dataset_name", data_config.get("dataset_name")),
        "data_view": SURROGATE_DATA_VIEW,
        "selected_client_ids": list(SURROGATE_SELECTED_CLIENT_IDS),
        "eval_subset": SURROGATE_EVAL_SUBSET,
        "validation_subset": SURROGATE_EVAL_SUBSET,
        "validation_examples": len(test_loader.dataset),
        "history": history,
        "validation_loss": float(test_loss),
        "validation_accuracy": final_val_acc,
        "validation_top1_accuracy": final_val_acc,
        "test_loss": float(test_loss),
        "test_accuracy": final_val_acc,
        "best_val_loss": float(best_val_loss),
        "best_val_accuracy": float(best_val_accuracy),
        "best_val_top1_accuracy": float(best_val_accuracy),
        "best_checkpoint_epoch": int(best_checkpoint_epoch),
        "best_checkpoint_val_accuracy": float(best_checkpoint_val_accuracy),
        "checkpoint_selection_metric": "lowest_val_loss",
        "last_train_accuracy": last_train_acc,
        "final_train_accuracy": last_train_acc,
        "final_train_top1_accuracy": last_train_acc,
        "final_train_val_accuracy_gap": train_val_gap,
        "final_train_val_top1_gap": train_val_gap,
        "epochs_completed": len(history["loss"]),
        "pool_size": SURROGATE_POOL_SIZE if SURROGATE_DATA_VIEW == "client_pool" else None,
        "batch_size": SURROGATE_BATCH_SIZE,
        "optimizer": deepcopy(SURROGATE_CFG.get("optimizer", {})),
        "scheduler": deepcopy(SURROGATE_CFG.get("scheduler", {})),
        "criterion": deepcopy(SURROGATE_CFG.get("criterion", {})),
        "model_kwargs": {"pretrained": SURROGATE_CFG.get("pretrained", True), "num_classes": int(num_classes)},
        "use_amp": use_amp,
    }
    return model, summary


## 5. Train and Save the Surrogate

Train MobileNetV2 and check whether the held-out accuracy is high enough for gradient attacks to be meaningful.


In [5]:
surrogate_model, surrogate_summary = train_surrogate_baseline()

print(
    f"Surrogate validation metrics -> loss: {surrogate_summary.get('validation_loss', surrogate_summary['test_loss']):.4f}, "
    f"accuracy: {surrogate_summary.get('validation_accuracy', surrogate_summary['test_accuracy']):.2f}%"
)

surrogate_summary


Epoch 1/300: train_loss=2.2594, train_acc=15.13%, val_loss=2.1998, val_acc=23.33%, lr=4.00e-04 - saved best checkpoint
Epoch 2/300: train_loss=2.0970, train_acc=42.89%, val_loss=1.9478, val_acc=74.48%, lr=1.12e-03 - saved best checkpoint
Epoch 3/300: train_loss=1.7413, train_acc=79.34%, val_loss=1.5135, val_acc=87.21%, lr=1.84e-03 - saved best checkpoint
Epoch 4/300: train_loss=1.2023, train_acc=87.63%, val_loss=0.9285, val_acc=90.52%, lr=2.56e-03 - saved best checkpoint
Epoch 5/300: train_loss=0.6506, train_acc=92.63%, val_loss=0.4646, val_acc=93.79%, lr=3.28e-03 - saved best checkpoint
Epoch 6/300: train_loss=0.3543, train_acc=95.00%, val_loss=0.2746, val_acc=95.01%, lr=4.00e-03 - saved best checkpoint
Epoch 7/300: train_loss=0.2193, train_acc=96.32%, val_loss=0.2073, val_acc=95.62%, lr=4.00e-03 - saved best checkpoint
Epoch 8/300: train_loss=0.1678, train_acc=97.11%, val_loss=0.1762, val_acc=96.18%, lr=4.00e-03 - saved best checkpoint
Epoch 9/300: train_loss=0.1232, train_acc=98.03%

{'training_mode': 'centralized',
 'surrogate_training_profile': 'tuned_imagenette',
 'profile_description': 'Longer target-aligned MobileNetV2 surrogate fine-tuning run.',
 'model': 'MobileNetV2Transfer',
 'dataset': 'Imagenette',
 'data_view': 'client_pool',
 'selected_client_ids': [0, 1, 2, 3],
 'eval_subset': 'selection',
 'validation_subset': 'selection',
 'validation_examples': 1963,
 'history': {'loss': [2.2593801423123008,
   2.097007347408094,
   1.7412970266844097,
   1.2023034340456913,
   0.6505562537594846,
   0.35426993715135674,
   0.2192672450291483,
   0.1677519162234507,
   0.12324330038145968,
   0.10425003582709715,
   0.09931426424729196,
   0.09492278099060059,
   0.08309872040623113,
   0.06414094967277427,
   0.05631277680789169,
   0.05828090165006487,
   0.038926240017539575,
   0.04808171297374524,
   0.029224150392569995,
   0.03271389752626419,
   0.02932256062171961,
   0.02729228682031757,
   0.027468745430049145,
   0.026499181691753238,
   0.024863062887

### Save and Validate the Surrogate

Save `module4_surrogate.json` and the surrogate checkpoint, then run basic health checks on loss, epoch count, and test accuracy.


In [6]:
surrogate_path = artifact_path("surrogate_metrics", "module4_surrogate.json")
with surrogate_path.open("w") as f:
    json.dump(surrogate_summary, f, indent=2)

surrogate_checkpoint_path = artifact_path("surrogate_checkpoint", "module4_surrogate.pt")
torch.save(surrogate_model.state_dict(), surrogate_checkpoint_path)

acc = float(surrogate_summary.get("validation_accuracy", surrogate_summary.get("test_accuracy", 0.0)))
loss = float(surrogate_summary.get("validation_loss", surrogate_summary.get("test_loss", float("nan"))))
history_len = int(surrogate_summary.get("epochs_completed", 0))
if not np.isfinite(loss):
    raise ValueError("Surrogate loss is not finite; revisit training settings.")
if history_len <= 0:
    raise ValueError("Surrogate training history is empty.")
if acc < 5.0:
    raise ValueError(f"Surrogate accuracy {acc:.2f}% is suspiciously low; revisit training settings.")
print(f"Surrogate validation accuracy: {acc:.2f}%")
print(f"Saved details to {surrogate_path.resolve()}")
print(f"Saved checkpoint to {surrogate_checkpoint_path.resolve()}")


Surrogate validation accuracy: 96.79%
Saved details to /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_surrogate.json
Saved checkpoint to /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_surrogate.pt


### Surrogate Training Curves

Plot the surrogate's training and validation curves. These curves help explain whether later transfer results come from a reasonable surrogate or from an underfit or overfit one.


In [7]:
def plot_surrogate_history(summary: dict) -> None:
    history = summary.get("history", {})
    if not history.get("loss"):
        print("No surrogate history to plot.")
        return
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["loss"], marker="o", label="train")
    axes[0].plot(epochs, history.get("val_loss", []), marker="s", label="val")
    axes[0].set_title("Surrogate loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["accuracy"], marker="o", label="train")
    axes[1].plot(epochs, history.get("val_accuracy", []), marker="s", label="val")
    axes[1].set_title("Surrogate accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = artifact_path("surrogate_history_plot", "surrogate_history.png")
    plt.savefig(out_path, dpi=150)
    print(f"Saved {out_path.resolve()}")


plot_surrogate_history(surrogate_summary)


Saved /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/surrogate_history.png


## Handoff Artifacts

After this notebook runs, the focused attack notebooks expect these files in `artifacts/`:

| Artifact | Used by |
| --- | --- |
| `module4_surrogate_config_used.json` | Surrogate training recipe and resolved centralized profile |
| `module4_surrogate.json` | Surrogate quality check before attacks |
| `module4_surrogate.pt` | Random-noise, FGSM, PGD, and transfer evaluation |
| `surrogate_history.png` | Instructor/student training-curve inspection |
